In [1]:
import numpy as np
import time

from LG_flow import nn, optim, Tensor
from LG_flow.utils.data import DataLoader, Mnist, to_one_hot

# 一.训练与测试函数包装

In [2]:
def train(dataloader, model, loss_fn, optimizer):
    train_loss, correct = 0., 0.

    for i, (images, labels) in enumerate(dataloader):
        images = images.reshape(images.shape[0], -1) / 255.0
        images = Tensor(images)
        labels_one_hot = Tensor(to_one_hot(labels, num_classes=10))

        pred = model.forward(images)
        loss = loss_fn.forward(pred, labels_one_hot)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        correct += np.sum(np.argmax(pred.data, 1) == labels)
        train_loss += loss.data

    return train_loss / len(dataloader), correct / len(dataloader.dataset)

In [3]:
def test(dataloader, model, loss_fn):
    test_loss, correct = 0., 0.

    for i, (images, labels) in enumerate(dataloader):
        images = images.reshape(images.shape[0], -1) / 255.0
        images = Tensor(images)
        labels_one_hot = Tensor(to_one_hot(labels, num_classes=10))

        pred = model.forward(images)
        loss = loss_fn.forward(pred, labels_one_hot)

        correct += np.sum(np.argmax(pred.data, 1) == labels)
        test_loss += loss.data

    return test_loss / len(dataloader), correct / len(dataloader.dataset)

# 二. 定义模型

In [4]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()

        self.fc1 = nn.Linear(784, 256)
        self.act1 = nn.ReLU()
        self.fc2 = nn.Linear(256, 64)
        self.act2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.fc1.forward(x)
        x = self.act1.forward(x)
        x = self.fc2.forward(x)
        x = self.act2.forward(x)
        x = self.fc3.forward(x)
        return x

model = MLP()

# 三. 加载数据

In [5]:
train_dataset = Mnist(root="data/MNIST/raw", train=True)
test_dataset = Mnist(root="data/MNIST/raw", train=False)

In [6]:
train_dataloader = DataLoader(train_dataset, batch=16, shuffle=True, drop_last=False)
test_dataloader = DataLoader(test_dataset, batch=16, shuffle=False, drop_last=False)

# 四. 损失函数与优化器

In [7]:
loss_fn = nn.CrossEntropyLoss(reduction="sum")

In [8]:
optimizer = optim.SGD(model.parameters(), lr=0.001)

# 五.训练

In [9]:
print(f"| {'':^6s} | {'time':^15s} | {'loss':^15s} | {'acc':^15s} |")
print(f"| {'epoch':^6s} | {'train':^6s} | {'test':^6s} | {'train':^6s} | {'test':^6s} | {'train':^6s} | {'test':^6s} |")

for epoch in range(50):

    time1 = time.time()
    train_loss, train_correct = train(train_dataloader, model, loss_fn, optimizer)
    time2 = time.time()
    test_loss, test_correct = test(test_dataloader, model, loss_fn)
    time3 = time.time()

    print(f"| {epoch:6d} | {time2-time1:.4f} | {time3-time2:.4f} | {train_loss:.4f} | {test_loss:.4f} | {train_correct:.4f} | {test_correct:.4f} |")


|        |      time       |      loss       |       acc       |
| epoch  | train  |  test  | train  |  test  | train  |  test  |
|      0 | 6.3380 | 0.1776 | 0.3554 | 0.2049 | 0.8995 | 0.9400 |
|      1 | 6.3726 | 0.1718 | 0.1682 | 0.1374 | 0.9506 | 0.9594 |
|      2 | 6.4092 | 0.1725 | 0.1226 | 0.1132 | 0.9638 | 0.9653 |
|      3 | 6.7338 | 0.2005 | 0.0970 | 0.0992 | 0.9711 | 0.9697 |
|      4 | 7.2772 | 0.1980 | 0.0796 | 0.0900 | 0.9768 | 0.9720 |
|      5 | 7.3061 | 0.1950 | 0.0650 | 0.0822 | 0.9809 | 0.9735 |
|      6 | 7.4034 | 0.1965 | 0.0554 | 0.0818 | 0.9838 | 0.9753 |
|      7 | 7.2726 | 0.1945 | 0.0467 | 0.0763 | 0.9865 | 0.9771 |
|      8 | 7.2599 | 0.1991 | 0.0399 | 0.0778 | 0.9886 | 0.9752 |
|      9 | 7.2568 | 0.1968 | 0.0340 | 0.0710 | 0.9909 | 0.9786 |
|     10 | 7.2897 | 0.1988 | 0.0286 | 0.0690 | 0.9920 | 0.9787 |
|     11 | 7.2657 | 0.1953 | 0.0254 | 0.0677 | 0.9936 | 0.9788 |
|     12 | 7.2919 | 0.1981 | 0.0215 | 0.0694 | 0.9951 | 0.9797 |
|     13 | 7.3743 | 0.197